# Task 2 Demo - Multi-Method Clustering

In this notebook I demonstrate the clustering part of the assignment using multiple methods.

What I show:

1. Build clustering features from normalized text.
2. Run at least two clustering methods (I run three: K-Means, Agglomerative, Spectral).
3. Compare methods across different cluster counts (max 10).
4. Evaluate with silhouette score and export results.

Reader guide:

- This notebook is written as a reproducible assignment walkthrough.
- Every major step writes an output CSV to `data/results/`.
- I use one sparse TF-IDF view (for interpretation) and one dense LSA view (for methods that prefer dense space).
- At the end, I extract common cluster terms and visualize them so the clusters can be inspected qualitatively.

In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.metrics import silhouette_score

from data_mining_assignment.core.data_io import ArticleDataset
from data_mining_assignment.tasks.preprocessing import TextNormalizer, TextPreprocessor

## Load data and build clustering features

Goal of this step:

- Load raw article text in original row order.
- Build two feature spaces from the same normalized text:
  - `clustering_feature_matrix`: sparse TF-IDF for interpretability.
  - `lsa_feature_matrix`: dense LSA projection for methods that work better in dense vectors.

Why this matters:

- TF-IDF keeps direct word mapping, which is needed for top-term analysis.
- LSA helps reduce sparsity and often improves distance-based behavior.

In [ ]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
# Keep row order untouched so labels stay aligned with doc_id.
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

text_normalizer = TextNormalizer()
# Build both text views in one pass (semantic view + structure-aware view).
normalized_bundle = text_normalizer.normalize_for_both_tasks(articles_data_frame["text"].tolist())

# Word-level TF-IDF for interpretable clustering.
# This matrix is used for K-Means and for top-term extraction later.
clustering_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf",
    max_features=20000,
    min_document_frequency=1,
    max_document_frequency=0.92,
    ngram_range=(1, 2),
    analyzer_mode="word",
)

clustering_feature_matrix = clustering_preprocessor.fit_transform(normalized_bundle.clustering_texts)

# Dense LSA view for methods that work better in dense space.
# Same source text, but projected into lower-dimensional dense vectors.
lsa_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=20000,
    min_document_frequency=1,
    max_document_frequency=0.92,
    ngram_range=(1, 2),
    analyzer_mode="word",
    dense_embedding_dimension=128,
    random_seed=42,
)
lsa_feature_matrix = lsa_preprocessor.fit_transform(normalized_bundle.clustering_texts)

clustering_feature_matrix.shape, lsa_feature_matrix.shape

## Compare methods and cluster counts

Goal of this step:

- Evaluate three clustering methods (`kmeans`, `agglomerative`, `spectral`).
- Test each method for `k` in `[4..10]` (assignment upper bound respected).
- Score each run with silhouette score.

How to read this section:

- Higher silhouette score usually means better separation and cohesion.
- Spectral can fail for some settings; failures are stored instead of stopping the notebook.
- All raw comparison results are exported to `notebook_05_clustering_method_metrics.csv`.

In [ ]:
candidate_cluster_counts = [4, 5, 6, 7, 8, 9, 10]

# Store one row per (method, k) for reproducible comparison.
scored_rows: list[dict[str, float | int | str]] = []
# Keep predicted labels so we can reuse the best run later without recomputing.
labels_by_method_and_k: dict[tuple[str, int], np.ndarray] = {}

for cluster_count in candidate_cluster_counts:
    # K-Means on sparse TF-IDF.
    kmeans_model = KMeans(n_clusters=cluster_count, n_init="auto", random_state=42)
    kmeans_labels = kmeans_model.fit_predict(clustering_feature_matrix)
    kmeans_silhouette = silhouette_score(clustering_feature_matrix, kmeans_labels)
    scored_rows.append(
        {
            "method": "kmeans",
            "cluster_count": cluster_count,
            "silhouette_score": float(kmeans_silhouette),
        }
    )
    labels_by_method_and_k[("kmeans", cluster_count)] = kmeans_labels

    # Agglomerative on dense LSA vectors with cosine metric.
    agglomerative_model = AgglomerativeClustering(n_clusters=cluster_count, metric="cosine", linkage="average")
    agglomerative_labels = agglomerative_model.fit_predict(lsa_feature_matrix)
    agglomerative_silhouette = silhouette_score(lsa_feature_matrix, agglomerative_labels)
    scored_rows.append(
        {
            "method": "agglomerative",
            "cluster_count": cluster_count,
            "silhouette_score": float(agglomerative_silhouette),
        }
    )
    labels_by_method_and_k[("agglomerative", cluster_count)] = agglomerative_labels

    try:
        # Spectral can be sensitive to graph settings, so keep it in try/except.
        spectral_model = SpectralClustering(
            n_clusters=cluster_count,
            affinity="nearest_neighbors",
            n_neighbors=12,
            assign_labels="kmeans",
            random_state=42,
        )
        spectral_labels = spectral_model.fit_predict(lsa_feature_matrix)
        spectral_silhouette = silhouette_score(lsa_feature_matrix, spectral_labels)
        scored_rows.append(
            {
                "method": "spectral",
                "cluster_count": cluster_count,
                "silhouette_score": float(spectral_silhouette),
            }
        )
        labels_by_method_and_k[("spectral", cluster_count)] = spectral_labels
    except Exception as spectral_error:
        # Log failure details but continue notebook execution.
        scored_rows.append(
            {
                "method": "spectral",
                "cluster_count": cluster_count,
                "silhouette_score": np.nan,
                "error": str(spectral_error),
            }
        )

# Sort rows for easier manual reading.
clustering_metrics_data_frame = pd.DataFrame(scored_rows).sort_values(
    ["method", "silhouette_score"], ascending=[True, False]
)
clustering_metrics_data_frame.to_csv(results_data_directory_path / "notebook_05_clustering_method_metrics.csv", index=False)
clustering_metrics_data_frame.head(15)

### Metric interpretation notes

- Compare rows **within each method** first (best `k` for that method).
- Then compare method winners against each other.
- If two setups are close numerically, use the term plots later to decide which one is more meaningful.

## Save best label set per method

Goal of this step:

- Pick the best `k` for each method using silhouette score.
- Export labels in original document order so outputs can be joined back to `doc_id`.

Exports:

- `notebook_05_best_method_labels.csv`: one column per best method label set.
- `notebook_05_best_method_summary.csv`: selected best score and `k` per method.

In [ ]:
best_rows = (
    clustering_metrics_data_frame.dropna(subset=["silhouette_score"])
    .sort_values(["method", "silhouette_score"], ascending=[True, False])
    .groupby("method", as_index=False)
    .first()
)

# Start with IDs to keep output directly traceable.
best_label_table = pd.DataFrame({"doc_id": articles_data_frame["doc_id"].tolist()})

for _, row in best_rows.iterrows():
    method_name = str(row["method"])
    selected_cluster_count = int(row["cluster_count"])
    # Reuse stored labels from earlier loop.
    selected_labels = labels_by_method_and_k[(method_name, selected_cluster_count)]
    best_label_table[f"{method_name}_k{selected_cluster_count}"] = selected_labels

best_label_table.to_csv(results_data_directory_path / "notebook_05_best_method_labels.csv", index=False)
best_rows.to_csv(results_data_directory_path / "notebook_05_best_method_summary.csv", index=False)

best_rows

## Cluster size diagnostics for each best method

Goal of this step:

- Check whether any best-method solution creates very tiny or very dominant clusters.

Why this matters:

- Silhouette alone can hide imbalanced partitions.
- Size diagnostics help detect unstable or overly broad clusters.

Export:

- `notebook_05_cluster_size_table.csv`

In [11]:
cluster_size_rows: list[dict[str, int | str]] = []

for _, row in best_rows.iterrows():
    method_name = str(row["method"])
    selected_cluster_count = int(row["cluster_count"])
    selected_labels = labels_by_method_and_k[(method_name, selected_cluster_count)]
    # Count docs per cluster label.
    label_counts = pd.Series(selected_labels, dtype="int64").value_counts().sort_index()
    for label_value, label_count in label_counts.items():
        cluster_size_rows.append(
            {
                "method": method_name,
                "cluster_count": selected_cluster_count,
                "label": label_value,
                "document_count": int(label_count),
            }
        )

cluster_size_table = pd.DataFrame(cluster_size_rows)
cluster_size_table.to_csv(results_data_directory_path / "notebook_05_cluster_size_table.csv", index=False)
cluster_size_table.head(20)

,method,cluster_count,label,document_count
0,agglomerative,5,0,2099
1,agglomerative,5,1,50
2,agglomerative,5,2,8
3,agglomerative,5,3,3
4,agglomerative,5,4,4
5,kmeans,8,0,70
6,kmeans,8,1,880
7,kmeans,8,2,112
8,kmeans,8,3,298
9,kmeans,8,4,192


## Common terms per cluster and visualization

Goal of this step:

- Make cluster meaning visible by extracting top terms per cluster.
- Plot readable per-cluster bar charts for each best method.

How common terms are computed:

- For each cluster, compute the **mean TF-IDF** vector of all cluster documents.
- Rank terms by descending mean TF-IDF.
- Keep top 12 terms (top 10 shown in the graph).

Export:

- `notebook_05_cluster_top_terms.csv`

In [ ]:
# Get TF-IDF feature names once for term lookup.
feature_name_list = clustering_preprocessor.get_feature_names()

cluster_top_term_rows: list[dict[str, int | float | str]] = []

for _, row in best_rows.iterrows():
    method_name = str(row["method"])
    selected_cluster_count = int(row["cluster_count"])
    selected_labels = labels_by_method_and_k[(method_name, selected_cluster_count)]

    unique_label_values = pd.Series(selected_labels, dtype="int64").unique().tolist()
    for label_value in sorted(unique_label_values):
        # Select docs belonging to current cluster.
        matching_document_mask = selected_labels == label_value

        # Use mean TF-IDF to find words that define this cluster.
        cluster_mean_vector = np.asarray(clustering_feature_matrix[matching_document_mask].mean(axis=0)).ravel()
        top_term_indices = np.argsort(cluster_mean_vector)[::-1][:12]

        for rank_value, term_index in enumerate(top_term_indices, start=1):
            cluster_top_term_rows.append(
                {
                    "method": method_name,
                    "cluster_count": selected_cluster_count,
                    "label": int(label_value),
                    "rank": int(rank_value),
                    "term": str(feature_name_list[term_index]),
                    "mean_tfidf": float(cluster_mean_vector[term_index]),
                }
            )

cluster_top_terms_table = pd.DataFrame(cluster_top_term_rows)
cluster_top_terms_table.to_csv(results_data_directory_path / "notebook_05_cluster_top_terms.csv", index=False)

# Print top terms for quick reading in plain text.
for _, row in best_rows.iterrows():
    method_name = str(row["method"])
    selected_cluster_count = int(row["cluster_count"])
    print(f"\nTop terms for {method_name} (k={selected_cluster_count})")

    method_rows = cluster_top_terms_table[
        (cluster_top_terms_table["method"] == method_name)
        & (cluster_top_terms_table["cluster_count"] == selected_cluster_count)
    ].sort_values(["label", "rank"])

    print(method_rows[["label", "rank", "term", "mean_tfidf"]].head(48).to_string(index=False))

# Plot per method with one bar chart per cluster.
for _, row in best_rows.iterrows():
    method_name = str(row["method"])
    selected_cluster_count = int(row["cluster_count"])

    method_rows = cluster_top_terms_table[
        (cluster_top_terms_table["method"] == method_name)
        & (cluster_top_terms_table["cluster_count"] == selected_cluster_count)
    ]

    unique_labels = sorted(method_rows["label"].unique().tolist())

    # One subplot per cluster to make term comparison clean.
    figure, axis_array = plt.subplots(
        nrows=len(unique_labels),
        ncols=1,
        figsize=(10, max(3.5, 2.8 * len(unique_labels))),
        constrained_layout=True,
    )
    if len(unique_labels) == 1:
        axis_array = [axis_array]

    for axis, label_value in zip(axis_array, unique_labels):
        cluster_rows = method_rows.loc[method_rows["label"] == label_value, :].sort_values(
            by="mean_tfidf", ascending=True
        )

        # Keep bars short so plot stays readable.
        cluster_rows = cluster_rows.tail(10)

        axis.barh(cluster_rows["term"], cluster_rows["mean_tfidf"], color="#4C78A8")
        axis.set_title(f"Cluster {label_value} top terms")
        axis.set_xlabel("Mean TF-IDF")
        axis.set_ylabel("Term")

    figure.suptitle(f"{method_name} common terms (k={selected_cluster_count})", fontsize=13)
    plt.show()


### Final reading notes

How to use these outputs in your report:

- Use `notebook_05_clustering_method_metrics.csv` for quantitative comparison.
- Use `notebook_05_cluster_size_table.csv` to discuss balance and stability.
- Use `notebook_05_cluster_top_terms.csv` + bar plots for qualitative interpretation.
- Pick the final method by combining numeric score + semantic coherence.

## Final conclusion (best method from this notebook)

This section gives a direct notebook conclusion for the report.

Selection rule:

1. Pick the highest silhouette score across all tested methods and `k` values.
2. Add a quick cluster-balance check using min and max cluster sizes.

This makes the recommendation transparent and reproducible.

In [ ]:
overall_best_row = (
    clustering_metrics_data_frame.dropna(subset=["silhouette_score"])
    .sort_values("silhouette_score", ascending=False)
    .iloc[0]
)

best_method_name = str(overall_best_row["method"])
best_cluster_count = int(overall_best_row["cluster_count"])
best_silhouette_score = float(overall_best_row["silhouette_score"])

best_labels = labels_by_method_and_k[(best_method_name, best_cluster_count)]
size_series = pd.Series(best_labels, dtype="int64").value_counts().sort_index()
min_cluster_size = int(size_series.min())
max_cluster_size = int(size_series.max())

conclusion_table = pd.DataFrame(
    [
        {
            "best_method": best_method_name,
            "best_cluster_count": best_cluster_count,
            "best_silhouette_score": best_silhouette_score,
            "min_cluster_size": min_cluster_size,
            "max_cluster_size": max_cluster_size,
        }
    ]
)
conclusion_table.to_csv(results_data_directory_path / "notebook_05_final_conclusion.csv", index=False)

print(
    f"Best method: {best_method_name} (k={best_cluster_count}) with silhouette={best_silhouette_score:.4f}. "
    f"Cluster sizes range from {min_cluster_size} to {max_cluster_size}."
)
conclusion_table
